# Encoder–decoder Transformer with RoPE for legal continuation

## 1. Hugging Face Login

In [1]:
%pip install -q huggingface_hub ijson

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


## 2. Check GPU

In [3]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


## 4. Load Tokenizer

In [ ]:
from collections import defaultdict, Counter
import heapq

SPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]

class BytelevelBPE:

    def __init__(self, vocab_size=5000):
        self.vocab_size = vocab_size
        self.specials = SPECIALS.copy()
        self.byte_vocab = [str(i) for i in range(256)]
        self.merges = []
        self.merge_ranks = {}
        self.vocab = []
        self.stoi = {}
        self.itos = {}
        # Maps each BPE symbol to the bytes represented by it
        self.symbol_to_bytes = {}
    
    # 1. WORD -> UTF-8 BYTE SYMBOLS

    def word_to_symbols(self, word):
        byte_seq = word.encode("utf-8")
        return [str(b) for b in byte_seq] + ["</w>"]

    # 2. BUILD WORD FREQUENCIES

    def build_word_freqs(self, corpus):
        word_counter = Counter()
        print("Building word frequencies...")
        for text in corpus:
            if not isinstance(text, str):
                continue

            word_counter.update(text.split())
        print(
            f"Unique whitespace-separated words: "
            f"{len(word_counter):,}"
        )
        # Convert each unique word to byte symbols only once
        word_freqs = {}

        for word, freq in word_counter.items():
            symbols = tuple(self.word_to_symbols(word))
            word_freqs[symbols] = (
                word_freqs.get(symbols, 0) + freq
            )

        return word_freqs


    # 3. PAIRS OCCURRING IN ONE WORD

    @staticmethod
    def get_pairs(symbols):
        return {
            (symbols[i], symbols[i + 1])
            for i in range(len(symbols) - 1)
        }

    # 4. COUNT PAIR OCCURRENCES IN ONE WORD

    @staticmethod
    def pair_counts_in_word(symbols):
        counts = Counter()
        for i in range(len(symbols) - 1):
            counts[
                (symbols[i], symbols[i + 1])
            ] += 1

        return counts

    # 5. MERGE A PAIR INSIDE ONE WORD

    @staticmethod
    def merge_symbols(symbols, pair, merged_symbol):
        new_symbols = []
        i = 0
        while i < len(symbols):
            if (
                i < len(symbols) - 1
                and symbols[i] == pair[0]
                and symbols[i + 1] == pair[1]
            ):
                new_symbols.append(merged_symbol)
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1

        return tuple(new_symbols)

    # 6. TRAIN OPTIMIZED BPE

    def train(self, corpus):
        word_freqs = self.build_word_freqs(corpus)
        vocab = (
            self.specials
            + self.byte_vocab
            + ["</w>"]
        )
        seen = set(vocab)
        for b in range(256):
            self.symbol_to_bytes[str(b)] = bytes([b])
        self.symbol_to_bytes["</w>"] = b""

        words = []
        frequencies = []
        for symbols, freq in word_freqs.items():
            words.append(symbols)
            frequencies.append(freq)
        del word_freqs

        pair_counts = defaultdict(int)
        pair_to_words = defaultdict(set)

        print("Building initial pair statistics...")

        for word_id, symbols in enumerate(words):
            freq = frequencies[word_id]
            local_counts = self.pair_counts_in_word(symbols)
            for pair, count in local_counts.items():
                pair_counts[pair] += count * freq
                pair_to_words[pair].add(word_id)

        print(
            f"Initial unique pairs: "
            f"{len(pair_counts):,}"
        )

        # MAX HEAP
        #
        # Python has min-heap, so use negative frequency.
        #
        # We keep stale entries and validate them when popped.
        # This avoids rebuilding the heap after every merge.

        heap = []
        for pair, count in pair_counts.items():
            heapq.heappush(heap,(-count, pair))
        target_merges = max(0,self.vocab_size - len(vocab))
        print(f"Target merges: {target_merges:,}")
        while (len(vocab) < self.vocab_size and heap):
            while heap:
                neg_count, pair = heapq.heappop(heap)
                heap_count = -neg_count
                current_count = pair_counts.get(pair,0)
                # Ignore stale heap entries
                if (current_count == heap_count and current_count > 0):
                    break
            else:
                break

            max_freq = pair_counts.get(pair, 0)

            # Same stopping condition as before
            if max_freq <= 1:
                break

            # Create collision-free merged symbol
            merged_symbol = (f"<bpe_{len(self.merges)}>")

            # Byte representation of new symbol
            left_bytes = self.symbol_to_bytes[pair[0]]
            right_bytes = self.symbol_to_bytes[pair[1]]
            self.symbol_to_bytes[merged_symbol] = left_bytes + right_bytes

            self.merges.append(pair)
            self.merge_ranks[pair] = (len(self.merges) - 1,merged_symbol)
            if merged_symbol not in seen:
                seen.add(merged_symbol)
                vocab.append(merged_symbol)

            # ----------------------------------------------------
            # IMPORTANT OPTIMIZATION
            #
            # Only process words containing selected pair.
            # ----------------------------------------------------

            affected_word_ids = list(pair_to_words.get(pair, set()))
            changed_pairs = set()
            for word_id in affected_word_ids:
                old_symbols = words[word_id]
                freq = frequencies[word_id]
                # Pair may have disappeared because of an earlier
                # update involving this word
                if pair not in self.get_pairs(old_symbols):
                    continue
                
                # Statistics BEFORE merge
                old_counts = self.pair_counts_in_word(old_symbols)

                # Merge selected pair
                new_symbols = self.merge_symbols(old_symbols,pair,merged_symbol)
                if new_symbols == old_symbols:
                    continue

                # Statistics AFTER merge
                new_counts = self.pair_counts_in_word(new_symbols)

                # Save updated word
                words[word_id] = new_symbols

                all_pairs = (
                    set(old_counts)
                    | set(new_counts)
                )

                # Update only pairs affected by this word

                for p in all_pairs:

                    old_n = old_counts.get(p, 0)
                    new_n = new_counts.get(p, 0)
                    delta = (
                        new_n - old_n
                    ) * freq

                    if delta != 0:
                        pair_counts[p] += delta
                        changed_pairs.add(p)

                    # Update inverted index
                    if new_n > 0:
                        pair_to_words[p].add(word_id)
                    else:
                        pair_to_words[p].discard(word_id)

            # Selected pair should now be gone from affected words
            pair_to_words[pair].clear()

            # Push updated frequencies into heap
            for p in changed_pairs:
                count = pair_counts.get(p, 0)
                if count > 0:
                    heapq.heappush(heap,(-count, p))

        self.vocab = vocab
        self.vocab_size = len(vocab)
        self.stoi = {
            symbol: i
            for i, symbol in enumerate(vocab)
        }

        self.itos = {
            i: symbol
            for symbol, i in self.stoi.items()
        }

        print("\nTokenizer training complete.")

        print(
            "Final vocabulary size:",
            self.vocab_size
        )

        print(
            "Number of merges:",
            len(self.merges)
        )

    # 7. APPLY BPE TO ONE WORD

    def apply_bpe(self, word):
        symbols = self.word_to_symbols(word)
        while len(symbols) > 1:
            best_pair = None
            best_rank = float("inf")
            best_merged_symbol = None

            for i in range(len(symbols) - 1):
                pair = (symbols[i],symbols[i + 1])
                merge_info = self.merge_ranks.get(pair)
                if merge_info is not None:
                    rank, merged_symbol = merge_info
                    if rank < best_rank:
                        best_rank = rank
                        best_pair = pair
                        best_merged_symbol = merged_symbol

            if best_pair is None:
                break

            symbols = list(
                self.merge_symbols(
                    tuple(symbols),
                    best_pair,
                    best_merged_symbol
                )
            )

        return symbols

    # 8. ENCODE

    def encode(self,text,add_bos=False,add_eos=False):
        ids = []
        if add_bos:
            ids.append(
                self.stoi["<s>"]
            )

        for word in text.split():
            symbols = self.apply_bpe(word)
            for symbol in symbols:
                ids.append(
                    self.stoi.get(
                        symbol,
                        self.stoi["<unk>"]
                    )
                )

        if add_eos:
            ids.append(
                self.stoi["</s>"]
            )

        return ids

    # 9. DECODE

    def decode(self, ids):

        words = []
        current_bytes = bytearray()

        for token_id in ids:
            symbol = self.itos.get(token_id)
            if symbol is None:
                continue
            if symbol in self.specials:
                continue

            # Determine whether this token ends a word
            ends_word = self._symbol_ends_word(symbol)
            byte_value = self.symbol_to_bytes.get(symbol, b"")
            current_bytes.extend(byte_value)

            if ends_word:
                try:
                    word = bytes(current_bytes).decode("utf-8")
                    words.append(word)
                except UnicodeDecodeError:
                    words.append("<unk>")
                current_bytes = bytearray()

        # Remaining bytes
        if current_bytes:
            try:
                words.append(bytes(current_bytes).decode("utf-8"))
            except UnicodeDecodeError:
                words.append("<unk>")

        return " ".join(words)

    # 10. CHECK WHETHER SYMBOL CONTAINS </w> AT END

    def _symbol_ends_word(self, symbol):
        if symbol == "</w>":
            return True
        # Find merge that created this symbol
        for pair, (_, merged) in self.merge_ranks.items():
            if merged == symbol:
                return self._symbol_ends_word(
                    pair[1]
                )

        return False

    # 11. VOCAB SIZE

    def get_vocab_size(self):
        return len(self.vocab)

## 9. Define The Model

In [5]:
import math
from dataclasses import dataclass, asdict
import torch
import torch.nn as nn
import torch.nn.functional as F

@dataclass
class TransformerConfig:
    vocab_size: int
    pad_id: int
    d_model: int = 512
    n_heads: int = 8
    d_ff: int = 2048
    n_layers: int = 6
    dropout: float = 0.1
    max_src_len: int = 2048
    max_tgt_len: int = 2048
    pos_emb_type: str = "rope"
    rope_base: float = 10000.0


class EmbeddingDropout(nn.Module):
    # Preserve the embedding dropout and configured length limit without adding PE.
    def __init__(self, max_len, dropout):
        super().__init__()
        self.max_len = max_len
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        if x.size(1) > self.max_len:
            raise ValueError('Sequence exceeds configured context length')
        return self.dropout(x)


class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, base=10000.0):
        super().__init__()
        if head_dim % 2:
            raise ValueError('RoPE requires an even head dimension')
        inv_freq = base ** (
            -torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim
        )
        self.register_buffer('inv_freq', inv_freq, persistent=False)

    def forward(self, x):
        # x: [batch, heads, length, head_dim]. Split-half pairs: (i, i + d/2).
        # Generation recomputes the full decoder sequence, so positions start at 0.
        with torch.autocast(device_type=x.device.type, enabled=False):
            positions = torch.arange(x.size(2), device=x.device, dtype=torch.float32)
            angles = torch.outer(positions, self.inv_freq.float())
            cos = angles.cos()[None, None, :, :]
            sin = angles.sin()[None, None, :, :]
        cos, sin = cos.to(x.dtype), sin.to(x.dtype)
        first, second = x.chunk(2, dim=-1)
        return torch.cat((first * cos - second * sin,
                          first * sin + second * cos), dim=-1)


In [6]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout, use_rope=False, rope_base=10000.0):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.rope = RotaryEmbedding(self.d_head, rope_base) if use_rope else None
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def split_heads(self, x):
        # [batch, length, d_model] -> [batch, heads, length, d_head]
        b, length, _ = x.shape
        return x.reshape(b, length, self.n_heads, self.d_head).transpose(1, 2)

    def forward(self, query, key, value, allowed, return_attention=False):
        q = self.split_heads(self.q_proj(query))
        k = self.split_heads(self.k_proj(key))
        v = self.split_heads(self.v_proj(value))
        if self.rope is not None:
            q = self.rope(q)
            k = self.rope(k)
        # True means this key may be attended to, including the causal mask.
        weights = None
        if return_attention:
            # Keep explicit probabilities for attention visualizations.
            with torch.autocast(device_type=q.device.type, enabled=False):
                scores = (q.float() @ k.float().transpose(-2, -1)) / math.sqrt(self.d_head)
                scores = scores.masked_fill(~allowed, float('-inf'))
                weights = scores.softmax(dim=-1)
            context = self.dropout(weights.to(v.dtype)) @ v
        else:
            # Let PyTorch select a supported backend; dropout must be 0 at eval.
            context = F.scaled_dot_product_attention(
                q, k, v, attn_mask=allowed,
                dropout_p=self.dropout.p if self.training else 0.0,
                is_causal=False,  # Causality is already present in allowed.
            )
        b, _, length, _ = context.shape
        context = context.transpose(1, 2).contiguous().reshape(b, length, -1)
        output = self.out_proj(context)
        return output, weights.mean(dim=1) if return_attention else None

In [7]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.linear2(F.relu(self.linear1(x)))

class EncoderLayer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.attention = MultiHeadAttention(
            cfg.d_model, cfg.n_heads, cfg.dropout, use_rope=True, rope_base=cfg.rope_base)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff)
        self.norm1 = nn.LayerNorm(cfg.d_model)
        self.norm2 = nn.LayerNorm(cfg.d_model)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x, source_mask):
        attended, _ = self.attention(x, x, x, source_mask)
        x = self.norm1(x + self.dropout(attended))
        x = self.norm2(x + self.dropout(self.ffn(x)))
        return x

class DecoderLayer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.self_attention = MultiHeadAttention(
            cfg.d_model, cfg.n_heads, cfg.dropout, use_rope=True, rope_base=cfg.rope_base)
        self.cross_attention = MultiHeadAttention(
            cfg.d_model, cfg.n_heads, cfg.dropout, use_rope=False)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff)
        self.norm1 = nn.LayerNorm(cfg.d_model)
        self.norm2 = nn.LayerNorm(cfg.d_model)
        self.norm3 = nn.LayerNorm(cfg.d_model)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x, memory, target_mask, source_mask, return_attention=False):
        attended, _ = self.self_attention(x, x, x, target_mask)
        x = self.norm1(x + self.dropout(attended))
        attended, weights = self.cross_attention(x, memory, memory, source_mask,
                                                  return_attention)
        x = self.norm2(x + self.dropout(attended))
        x = self.norm3(x + self.dropout(self.ffn(x)))
        return x, weights

In [8]:
class LegalTransformer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        if cfg.pos_emb_type != 'rope':
            raise ValueError('This architecture expects a RoPE configuration')
        self.cfg = cfg
        # Separate weights make the three roles easy to inspect.
        self.source_embedding = nn.Embedding(cfg.vocab_size, cfg.d_model, padding_idx=cfg.pad_id)
        self.target_embedding = nn.Embedding(cfg.vocab_size, cfg.d_model, padding_idx=cfg.pad_id)
        self.source_position = EmbeddingDropout(cfg.max_src_len, cfg.dropout)
        self.target_position = EmbeddingDropout(cfg.max_tgt_len, cfg.dropout)
        self.encoder = nn.ModuleList([EncoderLayer(cfg) for _ in range(cfg.n_layers)])
        self.decoder = nn.ModuleList([DecoderLayer(cfg) for _ in range(cfg.n_layers)])
        self.output = nn.Linear(cfg.d_model, cfg.vocab_size)
        for parameter in self.parameters():
            if parameter.dim() > 1:
                nn.init.xavier_uniform_(parameter)
        with torch.no_grad():
            self.source_embedding.weight[cfg.pad_id].zero_()
            self.target_embedding.weight[cfg.pad_id].zero_()

    def encode(self, source_ids):
        source_mask = source_ids.ne(self.cfg.pad_id)[:, None, None, :]
        x = self.source_embedding(source_ids) * math.sqrt(self.cfg.d_model)
        x = self.source_position(x)
        for layer in self.encoder:
            x = layer(x, source_mask)
        return x, source_mask

    def decode(self, target_ids, memory, source_mask, return_attention=False):
        length = target_ids.size(1)
        causal = torch.ones(length, length, dtype=torch.bool, device=target_ids.device).tril()
        valid_keys = target_ids.ne(self.cfg.pad_id)[:, None, None, :]
        target_mask = valid_keys & causal[None, None, :, :]
        x = self.target_embedding(target_ids) * math.sqrt(self.cfg.d_model)
        x = self.target_position(x)
        weights = None
        for index, layer in enumerate(self.decoder):
            x, weights = layer(x, memory, target_mask, source_mask,
                               return_attention and index == len(self.decoder) - 1)
        logits = self.output(x)
        return (logits, weights) if return_attention else logits

    def forward(self, source_ids, target_ids, return_attention=False):
        memory, source_mask = self.encode(source_ids)
        return self.decode(target_ids, memory, source_mask, return_attention)

# **Phase 6**

In [9]:
%pip install -q bert-score

Note: you may need to restart the kernel to use updated packages.


In [10]:
%pip install -q rouge-score

Note: you may need to restart the kernel to use updated packages.


In [11]:
import math
import json
import random
import re
import gc
import zipfile

from pathlib import Path
from dataclasses import asdict
from collections import defaultdict
from functools import lru_cache

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from torch.utils.checkpoint import checkpoint
from rouge_score import rouge_scorer


MODEL_DIR = Path("/kaggle/input/datasets/tejasvaidya137/model2-for-phase6-files")
OUTPUT_DIR = Path("/kaggle/working/phase6_encdec_lora")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42

RANK = 16
ALPHA = 32
ENCODER_ADAPT_LAST_N = 2

BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 2
LEARNING_RATE = 5e-4

EPOCHS = 10
PATIENCE = 3
USE_ACTIVATION_CHECKPOINTING = True

# Fixed subset for generation-based monitoring each epoch.
GENERATION_DEV_SIZE = 64

NO_REPEAT_NGRAM_SIZE = 6
MIN_NEW_TOKENS = 32

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

use_amp = device.type == "cuda"
amp_dtype = (
    torch.bfloat16
    if use_amp and torch.cuda.is_bf16_supported()
    else torch.float16
)

In [12]:
import hashlib

from huggingface_hub import hf_hub_download
from kaggle_secrets import UserSecretsClient


HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")

TRAIN_PATH = Path(
    hf_hub_download(
        repo_id="Exploration-Lab/CS-781-Capstone",
        filename="Phase_6_Summ/Train/train.json",
        repo_type="dataset",
        token=HF_TOKEN,
    )
)

raw_records = json.loads(
    TRAIN_PATH.read_text(encoding="utf-8")
)


def text_hash(text):
    normalized = " ".join(text.split())

    return hashlib.sha256(
        normalized.encode("utf-8")
    ).hexdigest()


records = []
seen_pairs = set()

for row in raw_records:
    document = row.get("document")
    summary = row.get("summary")

    if (
        not isinstance(document, str)
        or not isinstance(summary, str)
        or not document.strip()
        or not summary.strip()
    ):
        continue

    document = document.strip()
    summary = summary.strip()

    document_hash = text_hash(document)
    summary_hash = text_hash(summary)

    pair = (document_hash, summary_hash)

    if pair in seen_pairs:
        continue

    seen_pairs.add(pair)

    records.append({
        "id": str(row["id"]),
        "document": document,
        "summary": summary,
        "document_hash": document_hash,
    })

del raw_records

Phase_6_Summ/Train/train.json: reconstructing file:   0%|          |  0.00B /  211MB            

Phase_6_Summ/Train/train.json: downloading bytes:           |  0.00B            

In [13]:
state = json.loads(
    (MODEL_DIR / "tokenizer.json").read_text(encoding="utf-8")
)

tokenizer = BytelevelBPE(vocab_size=state["vocab_size"])

tokenizer.specials = state["specials"]
tokenizer.vocab = state["vocab"]
tokenizer.vocab_size = len(tokenizer.vocab)

tokenizer.stoi = {
    symbol: i for i, symbol in enumerate(tokenizer.vocab)
}
tokenizer.itos = {
    i: symbol for symbol, i in tokenizer.stoi.items()
}

tokenizer.merges = [tuple(pair) for pair in state["merges"]]

tokenizer.merge_ranks = {
    tuple(record["pair"]): (
        record["rank"],
        record["merged_symbol"],
    )
    for record in state["merge_ranks"]
}

tokenizer.symbol_to_bytes = {
    symbol: bytes(values)
    for symbol, values in state["symbol_to_bytes"].items()
}

# Cache word tokenization during passage selection.
tokenizer.apply_bpe = lru_cache(maxsize=100_000)(
    tokenizer.apply_bpe
)

# Faster decoding with the same word-boundary behavior.
word_ends = {"</w>": True}

for record in sorted(state["merge_ranks"], key=lambda x: x["rank"]):
    word_ends[record["merged_symbol"]] = word_ends.get(
        record["pair"][1], False
    )

tokenizer._symbol_ends_word = lambda symbol: word_ends.get(symbol, False)

PAD_ID = tokenizer.stoi["<pad>"]
BOS_ID = tokenizer.stoi["<s>"]
EOS_ID = tokenizer.stoi["</s>"]

del state

In [14]:
VAL_FRACTION = 0.10

source_hash = hashlib.sha256(
    TRAIN_PATH.read_bytes()
).hexdigest()

split_path = OUTPUT_DIR / "split.json"

if split_path.exists():
    split = json.loads(
        split_path.read_text(encoding="utf-8")
    )

    if split["source_sha256"] != source_hash:
        raise ValueError(
            "The training file differs from the saved split."
        )

else:
    parent = list(range(len(records)))

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    owners = {}

    # Keep matching IDs or documents in the same split.
    for i, row in enumerate(records):
        for key in (
            ("id", row["id"]),
            ("document", row["document_hash"]),
        ):
            if key in owners:
                parent[find(i)] = find(owners[key])
            else:
                owners[key] = i

    groups = [find(i) for i in range(len(records))]
    unique_groups = sorted(set(groups))

    random.Random(SEED).shuffle(unique_groups)

    n_dev = min(
        len(unique_groups) - 1,
        max(1, round(len(unique_groups) * VAL_FRACTION)),
    )

    dev_groups = set(unique_groups[:n_dev])

    split = {
        "seed": SEED,
        "source_sha256": source_hash,
        "train_indices": [
            i for i, group in enumerate(groups)
            if group not in dev_groups
        ],
        "dev_indices": [
            i for i, group in enumerate(groups)
            if group in dev_groups
        ],
    }

    split_path.write_text(
        json.dumps(split, indent=2),
        encoding="utf-8",
    )


train_indices = split["train_indices"]
dev_indices = split["dev_indices"]

print("Training examples:", len(train_indices))
print("Validation examples:", len(dev_indices))

Training examples: 6301
Validation examples: 700


In [15]:
class LoRA(nn.Module):
    def __init__(self, base, rank, alpha):
        super().__init__()

        self.base = base
        self.base.requires_grad_(False)
        self.scale = alpha / rank

        self.A = nn.Linear(base.in_features, rank, bias=False)
        self.B = nn.Linear(rank, base.out_features, bias=False)

        self.A.to(device=base.weight.device, dtype=base.weight.dtype)
        self.B.to(device=base.weight.device, dtype=base.weight.dtype)

        nn.init.normal_(self.A.weight, std=0.02)
        nn.init.zeros_(self.B.weight)

    def forward(self, x):
        return self.base(x) + self.scale * self.B(self.A(x))

In [16]:
class SummarizationModel(LegalTransformer):
    def encode(self, source_ids):
        source_mask = source_ids.ne(self.cfg.pad_id)[:, None, None, :]

        x = self.source_embedding(source_ids) * math.sqrt(self.cfg.d_model)
        x = self.source_position(x)

        for layer in self.encoder:
            use_checkpoint = (
                self.training
                and USE_ACTIVATION_CHECKPOINTING
                and (layer.adapted or x.requires_grad)
            )

            if use_checkpoint:
                x = checkpoint(
                    layer, x, source_mask,
                    use_reentrant=False,
                )
            else:
                x = layer(x, source_mask)

        return x, source_mask

    def decode(
        self,
        target_ids,
        memory,
        source_mask,
        return_attention=False,
        last_only=False,
    ):
        length = target_ids.size(1)

        causal = torch.ones(
            length, length,
            dtype=torch.bool,
            device=target_ids.device,
        ).tril()

        valid = target_ids.ne(self.cfg.pad_id)[:, None, None, :]
        target_mask = valid & causal[None, None]

        x = self.target_embedding(target_ids) * math.sqrt(self.cfg.d_model)
        x = self.target_position(x)

        weights = None

        for i, layer in enumerate(self.decoder):
            need_attention = return_attention and i == len(self.decoder) - 1

            if self.training and USE_ACTIVATION_CHECKPOINTING:
                x, weights = checkpoint(
                    layer,
                    x,
                    memory,
                    target_mask,
                    source_mask,
                    need_attention,
                    use_reentrant=False,
                )
            else:
                x, weights = layer(
                    x, memory, target_mask, source_mask, need_attention
                )

        logits = self.output(x[:, -1:] if last_only else x)

        return (logits, weights) if return_attention else logits

In [17]:
saved = torch.load(
    MODEL_DIR / "model.pt",
    map_location="cpu",
    weights_only=True,
)

cfg = TransformerConfig(**saved["model_config"])

model = SummarizationModel(cfg)
model.load_state_dict(saved["model_state_dict"], strict=True)
del saved

if cfg.vocab_size != tokenizer.get_vocab_size() or cfg.pad_id != PAD_ID:
    raise ValueError("Use the tokenizer saved with this checkpoint.")

model.requires_grad_(False)

ATTENTION_TARGETS = ["q_proj", "k_proj", "v_proj", "out_proj"]
FFN_TARGETS = ["linear1", "linear2"]


def adapt_attention(attention):
    for name in ATTENTION_TARGETS:
        setattr(
            attention,
            name,
            LoRA(getattr(attention, name), RANK, ALPHA),
        )


def adapt_ffn(ffn):
    for name in FFN_TARGETS:
        setattr(
            ffn,
            name,
            LoRA(getattr(ffn, name), RANK, ALPHA),
        )


encoder_start = max(0, len(model.encoder) - ENCODER_ADAPT_LAST_N)

for i, layer in enumerate(model.encoder):
    layer.adapted = i >= encoder_start

    if layer.adapted:
        adapt_attention(layer.attention)
        adapt_ffn(layer.ffn)

for layer in model.decoder:
    adapt_attention(layer.self_attention)
    adapt_attention(layer.cross_attention)
    adapt_ffn(layer.ffn)

model = model.to(device)

# Read actual limits from the saved checkpoint.
MAX_SOURCE_TOKENS = min(2048, cfg.max_src_len)
MAX_TARGET_TOKENS = min(2048, cfg.max_tgt_len)

print("Encoder source limit:", MAX_SOURCE_TOKENS)
print("Decoder target limit:", MAX_TARGET_TOKENS)

print(
    "Trainable parameters:",
    sum(p.numel() for p in model.parameters() if p.requires_grad),
)

Encoder source limit: 1024
Decoder target limit: 1024
Trainable parameters: 1572864


In [18]:
SECTION_SHARES = {
    "facts": 0.25,
    "issues": 0.10,
    "arguments": 0.15,
    "reasoning": 0.35,
    "decision": 0.15,
}

CUES = {
    "facts": r"\b(facts?|background|appellant was|respondent was|"
             r"prosecution case|on appeal|filed a suit)\b",

    "issues": r"\b(whether|question for consideration|"
              r"question of law|issues? for determination)\b",

    "arguments": r"\b(contended|submitted|argued|"
                 r"learned counsel|submission|contention)\b",

    "reasoning": r"\b(we hold|we find|we consider|"
                 r"in our view|in our opinion|it follows|"
                 r"therefore|interpretation|cannot accept)\b",

    "decision": r"\b(in the result|accordingly|set aside|"
                r"we allow|we dismiss)\b|"
                r"\b(appeal|petition|application)s?\b.{0,60}"
                r"\b(allowed|dismissed|disposed)\b",
}

CUES = {
    role: re.compile(pattern, re.I)
    for role, pattern in CUES.items()
}

HEADINGS = {
    "facts": "facts",
    "background": "facts",
    "factual background": "facts",
    "issues": "issues",
    "questions of law": "issues",
    "arguments": "arguments",
    "submissions": "arguments",
    "contentions": "arguments",
    "analysis": "reasoning",
    "discussion": "reasoning",
    "reasoning": "reasoning",
    "findings": "reasoning",
    "conclusion": "decision",
    "decision": "decision",
    "order": "decision",
}

GAP_IDS = tokenizer.encode("[...]")
SOURCE_CONTENT_BUDGET = MAX_SOURCE_TOKENS - 2

In [19]:
def build_passages(document):
    units = [
        " ".join(part.split())
        for part in re.split(r"\n+|(?<=[.!?])\s+(?=[A-Z(])", document)
        if part.strip()
    ]

    passages = []
    seen = set()
    heading_role = None

    for i, text in enumerate(units):
        heading = re.sub(
            r"^(?:\d+|[ivx]+)[.)]\s*",
            "",
            text,
            flags=re.I,
        ).strip(" :.-").lower()

        if heading in HEADINGS:
            heading_role = HEADINGS[heading]
            continue

        position = i / max(1, len(units) - 1)

        matched = {
            role: bool(pattern.search(text))
            for role, pattern in CUES.items()
        }

        if matched["decision"] and position >= 0.6:
            role = "decision"
        elif matched["issues"]:
            role = "issues"
        elif matched["arguments"]:
            role = "arguments"
        elif matched["reasoning"]:
            role = "reasoning"
        elif matched["facts"]:
            role = "facts"
        elif heading_role is not None:
            role = heading_role
        elif position < 0.25:
            role = "facts"
        elif position > 0.90:
            role = "decision"
        else:
            role = "reasoning"

        score = 1.0 + 2.0 * matched[role]
        score += float(heading_role == role)

        if role == "decision":
            score += 2 * position
        elif role == "facts":
            score += 1 - position

        # Downrank attribution boilerplate; do not delete decisions.
        if re.search(r"judgment.*delivered by", text, re.I):
            score -= 3

        # Keep long passages manageable without splitting words.
        words = text.split()

        for start in range(0, len(words), 60):
            piece = " ".join(words[start:start + 60])

            if piece in seen:
                continue

            seen.add(piece)

            passages.append({
                "role": role,
                "score": score,
                "ids": tokenizer.encode(piece),
            })

    return passages

In [20]:
def select_sections(document, document_ids=None):
    if document_ids is None:
        document_ids = tokenizer.encode(document)

    if len(document_ids) <= SOURCE_CONTENT_BUDGET:
        return [BOS_ID] + document_ids + [EOS_ID]

    passages = build_passages(document)
    selected = {}
    used = 0
    gap_cost = len(GAP_IDS)

    # First reserve coverage for each section.
    for role, fraction in SECTION_SHARES.items():
        quota = int(SOURCE_CONTENT_BUDGET * fraction)

        candidates = sorted(
            [
                i for i, passage in enumerate(passages)
                if passage["role"] == role
            ],
            key=lambda i: (-passages[i]["score"], i),
        )

        role_used = 0

        for i in candidates:
            ids = passages[i]["ids"]
            cost = len(ids) + gap_cost

            if role_used + cost <= quota:
                selected[i] = ids
                role_used += cost
                used += cost

        # Preserve some coverage if no complete passage fits.
        if role_used == 0 and candidates and quota > gap_cost:
            i = candidates[0]
            ids = passages[i]["ids"][:quota - gap_cost]
            selected[i] = ids
            used += len(ids) + gap_cost

    # Redistribute unused space.
    ranked = sorted(
        range(len(passages)),
        key=lambda i: (-passages[i]["score"], i),
    )

    for i in ranked:
        if i in selected:
            continue

        ids = passages[i]["ids"]
        cost = len(ids) + gap_cost

        if used + cost <= SOURCE_CONTENT_BUDGET:
            selected[i] = ids
            used += cost

    source = []

    for i in sorted(selected):
        if source:
            source.extend(GAP_IDS)
        source.extend(selected[i])

    return [BOS_ID] + source + [EOS_ID]

In [21]:
import inspect


tokenizer_hash = hashlib.sha256(
    (MODEL_DIR / "tokenizer.json").read_bytes()
).hexdigest()

selection_code = (
    inspect.getsource(build_passages)
    + inspect.getsource(select_sections)
)

cache_config = {
    "source_sha256": source_hash,
    "tokenizer_sha256": tokenizer_hash,
    "max_source_tokens": MAX_SOURCE_TOKENS,
    "max_target_tokens": MAX_TARGET_TOKENS,
    "section_shares": SECTION_SHARES,
    "headings": HEADINGS,
    "cues": {
        role: [pattern.pattern, pattern.flags]
        for role, pattern in CUES.items()
    },
    "selection_code_sha256": hashlib.sha256(
        selection_code.encode("utf-8")
    ).hexdigest(),
}

cache_path = OUTPUT_DIR / "selected_tokens.pt"

encoded_records = None

if cache_path.exists():
    cached = torch.load(
        cache_path,
        map_location="cpu",
        weights_only=True,
    )

    if cached["config"] == cache_config:
        encoded_records = cached["records"]

    del cached


if encoded_records is None:
    encoded_records = []

    for i, row in enumerate(records):
        document_ids = tokenizer.encode(row["document"])
        summary_ids = tokenizer.encode(row["summary"])

        source_ids = select_sections(
            row["document"],
            document_ids,
        )

        if len(summary_ids) + 1 <= MAX_TARGET_TOKENS:
            target_ids = summary_ids + [EOS_ID]
        else:
            target_ids = summary_ids[:MAX_TARGET_TOKENS]

        encoded_records.append({
            "source_ids": source_ids,
            "target_ids": target_ids,
        })

        if (i + 1) % 500 == 0:
            print(f"Prepared {i + 1}/{len(records)}")

    temporary = OUTPUT_DIR / "selected_tokens.tmp"

    torch.save(
        {
            "config": cache_config,
            "records": encoded_records,
        },
        temporary,
    )

    temporary.replace(cache_path)

print("Prepared examples:", len(encoded_records))

Prepared 500/7001
Prepared 1000/7001
Prepared 1500/7001
Prepared 2000/7001
Prepared 2500/7001
Prepared 3000/7001
Prepared 3500/7001
Prepared 4000/7001
Prepared 4500/7001
Prepared 5000/7001
Prepared 5500/7001
Prepared 6000/7001
Prepared 6500/7001
Prepared 7000/7001
Prepared examples: 7001


In [22]:
class SummaryDataset(Dataset):
    def __init__(self, indices):
        self.indices = list(indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        row = encoded_records[self.indices[index]]
        target = row["target_ids"]

        return {
            "source_ids": torch.tensor(row["source_ids"]),
            "decoder_ids": torch.tensor([BOS_ID] + target[:-1]),
            "labels": torch.tensor(target),
        }


def collate_summaries(rows):
    return {
        key: pad_sequence(
            [row[key] for row in rows],
            batch_first=True,
            padding_value=-100 if key == "labels" else PAD_ID,
        )
        for key in ("source_ids", "decoder_ids", "labels")
    }


train_loader = DataLoader(
    SummaryDataset(train_indices),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_summaries,
)

dev_loader = DataLoader(
    SummaryDataset(dev_indices),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_summaries,
)

In [23]:
@torch.inference_mode()
def generate_summary(source_ids):
    model.eval()

    source = torch.tensor(
        [source_ids],
        dtype=torch.long,
        device=device,
    )

    decoder_ids = torch.tensor(
        [[BOS_ID]],
        dtype=torch.long,
        device=device,
    )

    generated = []
    ngrams = defaultdict(set)
    n = NO_REPEAT_NGRAM_SIZE

    with torch.autocast(
        device_type=device.type,
        dtype=amp_dtype,
        enabled=use_amp,
    ):
        memory, source_mask = model.encode(source)

        for step in range(MAX_TARGET_TOKENS):
            logits = model.decode(
                decoder_ids,
                memory,
                source_mask,
                last_only=True,
            )[0, -1].float()

            logits[[PAD_ID, BOS_ID]] = float("-inf")

            if step < min(MIN_NEW_TOKENS, MAX_TARGET_TOKENS - 1):
                logits[EOS_ID] = float("-inf")

            if n > 1 and len(generated) >= n - 1:
                prefix = tuple(generated[-(n - 1):])
                banned = list(ngrams.get(prefix, ()))

                if banned:
                    logits[banned] = float("-inf")

            next_id = int(logits.argmax().item())

            if next_id == EOS_ID:
                break

            generated.append(next_id)

            if n > 1 and len(generated) >= n:
                ngrams[tuple(generated[-n:-1])].add(next_id)

            decoder_ids = torch.cat(
                [
                    decoder_ids,
                    torch.tensor([[next_id]], device=device),
                ],
                dim=1,
            )

    return tokenizer.decode(generated)

In [24]:
generation_dev_indices = random.Random(SEED).sample(
    dev_indices,
    min(GENERATION_DEV_SIZE, len(dev_indices)),
)

rouge = rouge_scorer.RougeScorer(
    ["rouge1", "rougeL"],
    use_stemmer=True,
)


def batch_loss(batch):
    source = batch["source_ids"].to(device)
    decoder = batch["decoder_ids"].to(device)
    labels = batch["labels"].to(device)

    logits = model(source, decoder)

    loss_sum = F.cross_entropy(
        logits.float().reshape(-1, cfg.vocab_size),
        labels.reshape(-1),
        ignore_index=-100,
        reduction="sum",
    )

    count = labels.ne(-100).sum()

    return loss_sum, count


@torch.inference_mode()
def evaluate_loss():
    model.eval()
    total_loss = 0.0
    total_tokens = 0

    for batch in dev_loader:
        with torch.autocast(
            device_type=device.type,
            dtype=amp_dtype,
            enabled=use_amp,
        ):
            loss_sum, count = batch_loss(batch)

        total_loss += loss_sum.item()
        total_tokens += count.item()

    return total_loss / total_tokens


def evaluate_generation(indices):
    predictions = []
    scores = []

    for index in indices:
        prediction = generate_summary(
            encoded_records[index]["source_ids"]
        )

        result = rouge.score(
            records[index]["summary"],
            prediction,
        )

        scores.append({
            name: result[name].fmeasure
            for name in ("rouge1", "rougeL")
        })

        predictions.append({
            "id": records[index]["id"],
            "reference": records[index]["summary"],
            "summary": prediction,
        })

    metrics = {
        name: float(np.mean([score[name] for score in scores]))
        for name in ("rouge1", "rougeL")
    }

    return metrics, predictions

In [25]:
trainable_parameters = [
    p for p in model.parameters() if p.requires_grad
]

optimizer = torch.optim.AdamW(
    trainable_parameters,
    lr=LEARNING_RATE,
    weight_decay=0.01,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=1,
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=use_amp and amp_dtype == torch.float16,
)

run_config = {
    "adapter": "lora",
    "rank": RANK,
    "alpha": ALPHA,
    "encoder_adapt_last_n": ENCODER_ADAPT_LAST_N,
    "decoder_adapt_last_n": len(model.decoder),
    "attention_targets": ATTENTION_TARGETS,
    "ffn_targets": FFN_TARGETS,
    "max_source_tokens": MAX_SOURCE_TOKENS,
    "max_target_tokens": MAX_TARGET_TOKENS,
    "section_shares": SECTION_SHARES,
    "no_repeat_ngram_size": NO_REPEAT_NGRAM_SIZE,
    "min_new_tokens": MIN_NEW_TOKENS,
    "generation_dev_indices": generation_dev_indices,
    "batch_size": BATCH_SIZE,
    "gradient_accumulation": GRAD_ACCUM_STEPS,
    "learning_rate": LEARNING_RATE,
    "seed": SEED,
}

(OUTPUT_DIR / "run_config.json").write_text(
    json.dumps(run_config, indent=2),
    encoding="utf-8",
)

1242

In [26]:
from bert_score import BERTScorer

# Use the second GPU when available.
bert_device = (
    "cuda:1"
    if torch.cuda.device_count() > 1
    else "cuda:0" if torch.cuda.is_available()
    else "cpu"
)

bert_scorer = BERTScorer(
    model_type="roberta-large",
    num_layers=17,
    lang="en",
    idf=False,
    rescale_with_baseline=False,
    device=bert_device,
    batch_size=4,
)

print("BERTScore device:", bert_device)
print("BERTScore settings:", bert_scorer.hash)

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore device: cuda:1
BERTScore settings: roberta-large_L17_no-idf_version=0.3.12(hug_trans=5.16.1)


In [27]:
def calculate_bertscore(predictions):
    candidates = [row["summary"] for row in predictions]
    references = [row["reference"] for row in predictions]

    precision, recall, f1 = bert_scorer.score(
        candidates,
        references,
        batch_size=4,
    )

    return {
        "bertscore_precision": precision.mean().item(),
        "bertscore_recall": recall.mean().item(),
        "bertscore_f1": f1.mean().item(),
    }

In [28]:
best_bertscore = -float("inf")
bad_epochs = 0
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()

    # Keep the fully frozen encoder prefix deterministic.
    model.source_position.eval()

    for layer in model.encoder[:encoder_start]:
        layer.eval()

    optimizer.zero_grad(set_to_none=True)

    total_loss = 0.0
    total_tokens = 0

    for step, batch in enumerate(train_loader):
        group_start = (step // GRAD_ACCUM_STEPS) * GRAD_ACCUM_STEPS

        group_size = min(
            GRAD_ACCUM_STEPS,
            len(train_loader) - group_start,
        )

        with torch.autocast(
            device_type=device.type,
            dtype=amp_dtype,
            enabled=use_amp,
        ):
            loss_sum, count = batch_loss(batch)
            loss = loss_sum / count

        scaler.scale(loss / group_size).backward()

        if (
            (step + 1) % GRAD_ACCUM_STEPS == 0
            or step + 1 == len(train_loader)
        ):
            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                trainable_parameters, 1.0
            )

            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        total_loss += loss_sum.item()
        total_tokens += count.item()

        if (step + 1) % 100 == 0:
            print(
                f"Epoch {epoch} | "
                f"Batch {step + 1}/{len(train_loader)} | "
                f"Loss: {total_loss / total_tokens:.4f}",
                flush=True,
            )

    dev_loss = evaluate_loss()

    metrics, predictions = evaluate_generation(
        generation_dev_indices
    )
    
    metrics.update(calculate_bertscore(predictions))
    
    current_score = metrics["bertscore_f1"]
    
    scheduler.step(current_score)
    
    improved = current_score > best_bertscore
    
    if improved:
        best_bertscore = current_score
        bad_epochs = 0
    else:
        bad_epochs += 1

    history.append({
        "epoch": epoch,
        "train_loss": total_loss / total_tokens,
        "dev_loss": dev_loss,
        **metrics,
    })

    saved = {
        "epoch": epoch,
        "model_config": asdict(cfg),
        "model_state_dict": model.state_dict(),
        "run_config": run_config,
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "best_bertscore": best_bertscore,
        "selection_metric": "bertscore_f1",
        "bertscore_hash": bert_scorer.hash,
        "bad_epochs": bad_epochs,
        "history": history,
    }

    temporary = OUTPUT_DIR / "latest.tmp"
    torch.save(saved, temporary)
    temporary.replace(OUTPUT_DIR / "latest.pt")

    if improved:
        torch.save(saved, OUTPUT_DIR / "best.pt")

        (OUTPUT_DIR / "best_dev_predictions.json").write_text(
            json.dumps(predictions, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )

    (OUTPUT_DIR / "history.json").write_text(
        json.dumps(history, indent=2),
        encoding="utf-8",
    )

    print(
        f"Epoch {epoch} | "
        f"Dev loss: {dev_loss:.4f} | "
        f"BERTScore F1: {metrics['bertscore_f1']:.4f} | "
        f"ROUGE-1: {metrics['rouge1']:.4f} | "
        f"ROUGE-L: {metrics['rougeL']:.4f}",
        flush=True,
    )

    if bad_epochs >= PATIENCE:
        print("Early stopping.")
        break

del saved

Epoch 1 | Batch 100/1576 | Loss: 3.9111
Epoch 1 | Batch 200/1576 | Loss: 3.8764
Epoch 1 | Batch 300/1576 | Loss: 3.8614
Epoch 1 | Batch 400/1576 | Loss: 3.8452
Epoch 1 | Batch 500/1576 | Loss: 3.8393
Epoch 1 | Batch 600/1576 | Loss: 3.8256
Epoch 1 | Batch 700/1576 | Loss: 3.8192
Epoch 1 | Batch 800/1576 | Loss: 3.8167
Epoch 1 | Batch 900/1576 | Loss: 3.8148
Epoch 1 | Batch 1000/1576 | Loss: 3.8125
Epoch 1 | Batch 1100/1576 | Loss: 3.8097
Epoch 1 | Batch 1200/1576 | Loss: 3.8044
Epoch 1 | Batch 1300/1576 | Loss: 3.8014
Epoch 1 | Batch 1400/1576 | Loss: 3.7971
Epoch 1 | Batch 1500/1576 | Loss: 3.7946
Epoch 1 | Dev loss: 3.6762 | BERTScore F1: 0.7838 | ROUGE-1: 0.2989 | ROUGE-L: 0.1893
Epoch 2 | Batch 100/1576 | Loss: 3.7240
Epoch 2 | Batch 200/1576 | Loss: 3.7504
Epoch 2 | Batch 300/1576 | Loss: 3.7526
Epoch 2 | Batch 400/1576 | Loss: 3.7469
Epoch 2 | Batch 500/1576 | Loss: 3.7436
Epoch 2 | Batch 600/1576 | Loss: 3.7418
Epoch 2 | Batch 700/1576 | Loss: 3.7360
Epoch 2 | Batch 800/1576 | L

In [29]:
saved = torch.load(
    OUTPUT_DIR / "best.pt",
    map_location="cpu",
    weights_only=True,
)

model.load_state_dict(saved["model_state_dict"])
del saved

model.eval()

for index in generation_dev_indices[:5]:
    prediction = generate_summary(
        encoded_records[index]["source_ids"]
    )

    print("\n" + "=" * 80)
    print("ID:", records[index]["id"])

    print("\nREFERENCE:\n")
    print(records[index]["summary"])

    print("\nGENERATED:\n")
    print(prediction)


ID: 411

REFERENCE:

Section 18 of the Court of Wards Act, 1879, provides that the Court of Wards "may sanction the giving of leases or farms of the whole or part of any property under its charge, and may direct the mortgage or sale of any part of such property, and may direct the doing of all such other acts as it may judge to be most for the benefit of the property and the advantage of the ward". In exercise of the power conferred by this section the Court of Wards sanctioned a deed of prospecting license in favour of B, the predecessor in interest of the appellant, and the same was executed on 26 3 1915. Subsequently, on 23 11 1917 the manager of the Court of Wards executed a deed modifying the terms of the deed dated 26 3 1915, by virtue of which the period of license could be extended up to 26 3 1951 under certain conditions. On 10 8 1937 the respondent having become major assumed management of the estate and thereafter repudiated the aforesaid deeds and contested their validity 

In [30]:
full_dev_metrics, full_dev_predictions = evaluate_generation(
    dev_indices
)

print(full_dev_metrics)

(OUTPUT_DIR / "full_dev_predictions.json").write_text(
    json.dumps(full_dev_predictions, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

{'rouge1': 0.32563014076615665, 'rougeL': 0.2039832214120706}


6060755

In [31]:
from huggingface_hub import hf_hub_download

test_path = hf_hub_download(
    repo_id="Exploration-Lab/CS-781-Capstone",
    filename="Phase_6_Summ/Test/test.json",
    repo_type="dataset",
    token=HF_TOKEN,
)

test_records = json.loads(
    Path(test_path).read_text(encoding="utf-8")
)

test_predictions = []

for i, row in enumerate(test_records):
    source_ids = select_sections(row["document"])

    test_predictions.append({
        "id": row["id"],
        "summary": generate_summary(source_ids),
    })

    if (i + 1) % 25 == 0:
        print(f"Generated {i + 1}/{len(test_records)}")

        (OUTPUT_DIR / "test_progress.json").write_text(
            json.dumps(test_predictions, ensure_ascii=False),
            encoding="utf-8",
        )


prediction_path = OUTPUT_DIR / "prediction.json"

prediction_path.write_text(
    json.dumps(test_predictions, ensure_ascii=False),
    encoding="utf-8",
)

with zipfile.ZipFile(
    OUTPUT_DIR / "submission.zip",
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    archive.write(prediction_path, arcname="prediction.json")

print("Saved:", OUTPUT_DIR / "submission.zip")

test.json:   0%|          | 0.00/2.73M [00:00<?, ?B/s]

Generated 25/100
Generated 50/100
Generated 75/100
Generated 100/100
Saved: /kaggle/working/phase6_encdec_lora/submission.zip
